<div class="alert alert-block alert-info">
<h3>Student Information</h3> Please provide information about yourself. We will NOT grade this submissing w/o all the information<br>
<b>Name</b>: Sanika Palande<br>
<b>NetID</b>: ssp214<br>
<b>Recitation (01,02,90,91)</b>: 02 <br>
<b>Notes to Grader</b> (optional):<br>
<br><br>
<b>IMPORTANT</b>
Your work will not be graded withour your initials below<br>
I certify that this lab represents my own work and I have read the RU academic intergrity policies at<br>
<a href="https://www.cs.rutgers.edu/academic-integrity/introduction">https://www.cs.rutgers.edu/academic-integrity/introduction </a><br>
<b>Initials</b>: SSP    


<h3>Grader Notes</h3>
<b>Your Grade<b>:<br>
<b>Grader Initials</b>:<br>
<b>Grader Comments</b> (optional):<br>
</div>

## CS 439 - Introduction to Data Science
### Fall 2021

# Lab 3: Data Cleaning and Visualization

### Due Date: Sunday October 10, 2021 by 11:59 PM ###

### Instructions
This lab is presented as a notebook. Please execute the cells that are already completed and your task is to fill in the code
between ### BEGIN SOLUTION ### and ### END SOLUTION ###. 

#### Important: Please do not add any new cells or change the order of cells. If you have questions, please contact the courseS staff.

In this lab, you will be working with a dataset from NYPD containing data on calls to the New York Police Department. Information about the datasets can be found https://opendata.cityofnewyork.us/

## Setup

Note that after activating matplotlib to display figures inline via the IPython magic `%matplotlib inline`, we configure a custom default figure size. Virtually every default aspect of matplotlib [can be customized](https://matplotlib.org/users/customizing.html).

In [ ]:
!pip install datascience

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 9)

# Part 1:Getting Data

We will work with the NYPD Historic complaint data set. Our first task is to estimate the size of this download by looking at the number of rows, columns and using an estimated size for a column (use a reasonable value). The site metadata is available from the page
https://data.cityofnewyork.us/Public-Safety/NYPD-Complaint-Data-Historic/qgea-i56i

In [ ]:
### BEGIN SOLUTION
estimated size of the download (based on metadata information from the site) =   7.38 Million Rows and 35 columns    
# please explain how you reached the answer

I looked on the website and utilized the metadata on given on there, which told me that there are 7.38 million rows
and 35 columns.

### END SOLUTION

### 1.1 Download the data
This file is large (use the estimate you did above). If it takes too long to download, you may want to interrupt and download the file using a browser and URL https://data.cityofnewyork.us/Public-Safety/NYPD-Complaint-Data-Historic/qgea-i56i

In [ ]:
import utils
data_dir = 'data'
data_url = 'https://data.cityofnewyork.us/Public-Safety/NYPD-Complaint-Data-Historic/qgea-i56i'

file_name = 'NYPD_Complaint_Data_Historic.csv'

# To retrieve the dataset, we will use the `utils.fetch_and_cache` utility from utils library. 
dest_path = utils.fetch_and_cache(data_url=data_url, file=file_name,data_dir=data_dir)
print(f'Located at {dest_path}')

### 1.2 Inspect the size of the file
It is helpful to get an idea of the size of the file. This can be done using functions in the utils library.

In [ ]:
# look at the size of the file w/o opening it using OS (https://docs.python.org/3/library/os.html). You can perform
# variety of operating system related functions from this package.
import os
### BEGIN SOLUTION
file_size = os.path.getsize('NYPD_Complaint_Data_Historic.csv')
print(file_size)
#### END SOLUTION

### 1.3 Split the large file
This data file NYPD_Complaint_Data_Historic.csv is too big to load into a single DataFrame. Let us split the large file into smaller files.  Let us find out the number of lines in the NYPD_Complaint_Data_Historic.csv file using utils.

In [ ]:
import utils

In [ ]:
# Using utils library, find the number of lines in the file
# import pysys
### BEGIN SOLUTION
linecount = utils.line_count('NYPD_Complaint_Data_Historic.csv')
print(linecount)
### END SOLUTION

In [ ]:
# Split the file into 10 smaller files. Estimate the number of lines in each file using the count above
# files should be created in the data folder and named NYPD_Complaint_Data_Historic_1.csv, 
# NYPD_Complaint_Data_Historic_2.csv, ... NYPD_Complaint_Data_Historic_10.csv etc
# It is possible that few lines from the original file may not be saved due to rounding errors.

### BEGIN SOLUTION
chunk_size = int(linecount/10)
i = 0
for chunk in pd.read_csv('NYPD_Complaint_Data_Historic.csv', chunksize = chunk_size):
    chunk.to_csv(f'NYPD_Complaint_Data_Historic_{i+1}.csv')
    print(utils.line_count(f'NYPD_Complaint_Data_Historic_{i+1}.csv'))
    i = i + 1


### END SOLUTION

### 1.4 Check the files in the data folder
Now, we'll use a method of the `Pathlib.Path` class called `glob` to list all files in the `data` directory. You will find useful information in pathlib [docs](https://docs.python.org/3/library/pathlib.html).

Below, we use pathlib's `glob` method to store the list of all files' names from the `data_dir` directory in the variable `file_names`. These names should be strings that contain only the file name (e.g. `dummy.txt` not `data/dummy.txt`). The asterisk (*) character is used with the `glob` method to match any string.

In [ ]:
from pathlib import Path
data_dir_path = Path('data') # creates a Path object that points to the data directory
file_names = [x.name for x in data_dir_path.glob('*') if x.is_file()]
file_names

### 1.5 Pre-processing of data
It is good to pre-process the data to see if the file can be opened in a Jupyter notebook. We need to avoid large files that can crash notebooks. Typically, files of size around 200 MB is ok to open into a DataFrame. In the following activities we will inspect the file w/o opening it as a DataFrame. Using utils.head 

In [ ]:
# Use the utils.head() to read up to 5 lines from the original file (w/o opening it)
from utils import head
utils.head('data/NYPD_Complaint_Data_Historic.csv')



### 1.6 Inspecting and describing data columns
There should be 35 columns in each record. Using header information and data types, describe the type of data in each
column. If you are unable to determine, just state so.
##### BEGIN SOLUTION
* CMPLNT_NUM : int64
* CMPLNT_FR_DT: datetime object
* CMPLNT_FR_TM: datetime object
* CMPLNT_TO_DT: datetime object
* CMPLNT_TO_TM: string object
* ADDR_PCT_CD: float64
* RPT_DT: datetime object
* KY_CD: int64
* OFNS_DESC: string object
* PD_CD: float64
* PD_DESC: string
* CRM_ATPT_CPTD_CD: string object
* LAW_CAT_CD: string object
* BORO_NM: string object
* LOC_OF_OCCUR_DESC: string object
* PREM_TYP_DESC: string object
* JURIS_DESC: string object
* JURISDICTION_CODE: float64
* PARKS_NM: string object
* HADEVELOPT: string object
* HOUSING_PSA: float64
* X_COORD_CD: float64
* Y_COORD_CD: float64
* SUSP_AGE_GROUP: string object
* SUSP_RACE:string object
* SUSP_SEX:string object
* TRANSIT_DISTRICT: float64
* Latitude: float64
* Longitude: float64
* Lat_Lon: object
* PATROL_BORO: string object
* STATION_NAME:string object
* VIC_AGE_GROUP: string object
* VIC_RACE: string object
* VIC_SEX: string object

##### END SOLUTION

# Part 2 - Exploratory Data Analysis
Exploratory data analysis (EDA) is the process of examining a subest of a large data set to see what we can know about the data. First we will explore one file NYPD_Complaint_Data_Historic_1.csv to see what we can find out.

### 2.1 Loading Data into a DataFrame
Load the first CSV file, NYPD_Complaint_Data_Historic_1.csv into a `pandas.DataFrame` object. Also do a time analysis to see how long it took to load the data into a DataFrame. Time should be printed in seconds. The time libraries https://docs.python.org/3/library/time.html can help.

In [ ]:
!pip install memory_profiler

In [ ]:
### BEGIN SOLUTION

import time

t = time.process_time()
df = pd.read_csv ('NYPD_Complaint_Data_Historic_1.csv')
df = df.drop('Unnamed: 0', 1)
elapsed_time = time.process_time() - t

print(elapsed_time)





### END SOLUTION

### 2.2 Description of Fields
Let's also check some basic information about these files using the `DataFrame.describe` and `DataFrame.info` methods. Describe columns that can be removed based on the information.

In [ ]:
### BEGIN SOLUTION
df.describe()
df.info()

# What columns can be removed from the DataFrame? A reasonable rule of thumb is that if a column is missing more than 
# 50% of the data then it should be removed

    #PARKS_NM, HADEVELOPT, HOUSING_PSA, TRANSIT_DISTRICT, STATION_NAME

### END SOLUTION

### 2.2 Finding Uniques

Notice that the functions above reveal type information for the columns, as well as some basic statistics about the numerical columns found in the DataFrame. However, we still need more information about what each column represents. Let's explore the data further.

find the number of unique values in each DataFrame column and answer the questions below.

In [ ]:
### BEGIN SOLUTION
print(df.nunique())

# Questions
# 1. How many distinct locations where the complaints have come from? 5  
# 2. How many age groups are represented in the data set? 44 
# 3. How many boroughs are included in the data set? 5
# 4. How many offense types are listed in this data set? 65

### END SOLUTION

### 2.3 Offense by Boro
Using GroupBy operation, create a DataFrame that groups offenses by Boro. call the DataFrame calls_by_Boro_and_offense

In [ ]:
### BEGIN SOLUTION
calls_by_Boro_and_offense = df.groupby('BORO_NM').groups
calls_by_Boro_and_offense

### END SOLUTION

### 2.4 Offenses in Bronx

In the cell below, find a list of strings corresponding to the possible values for `OFNS_DESC` when `BORO` is "BRONX". Create an expression that automatically extracts the names of the offenses.

In [ ]:
### BEGIN SOLUTION

#df when boro is bronx
#bronx_df = df[df['BORO_NM'] == 'BRONX']

result = df[(df['BORO_NM'] == 'BRONX')]['OFNS_DESC'].unique()

print(result)

### END SOLUTION

# How many offenses were committed in Bronx during the analysis period? 

#print(len(calls_by_Boro_and_offense.get_group('BRONX')))

### 2.5 Most Common Crimes in NYC

What are the five crime types of OFNS_DESC that have the most crime events in Bronx? You may need to use `value_counts` to find the answer. Save your results as a list of strings.

**Hint:** *The `keys` method of the Series class might be useful.*

In [ ]:
### BEGIN SOLUTION
result = df[(df['BORO_NM'] == 'BRONX')]
ofns_ser = result['OFNS_DESC'].value_counts()
top_5_ofns_df = ofns_ser.head()
top_5_ofns_df.keys()

### END SOLUTION

### 2.6 Probability of a Crime in Bronx
What is the probability that a the crime "Arson" can happen in Bronx?

In [ ]:
### BEGIN SOLUTION

#dataframe with Bronx Only
result = df[(df['BORO_NM'] == 'BRONX')]

total_count = result['OFNS_DESC'].value_counts().sum()
ar_count = (result['OFNS_DESC'].values == 'ARSON').sum()

prob = (ar_count/total_count)
prob

### END SOLUTION

---
# Part 3: Visualizing the Data

## Pandas vs. Seaborn Plotting

Pandas offers basic functionality for plotting. For example, the `DataFrame` and `Series` classes both have a `plot` method. However, the basic plots generated by pandas are not particularly pretty. While it's possible to manually use matplotlib commands to make pandas plots look better, we'll instead use a high level plotting library called Seaborn that will take care of most of this for us.

As you learn to do data visualization, you may find the [pandas documentation](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.plot.html) and [Seaborn documentation](https://seaborn.pydata.org/api.html) helpful!

### 3.1 Plotting a Series
Using the built-in plotting functionality of pandas, such as `plot` method of the `Series` class to generate a `barh` plot type,  display the value counts for `OFNS_DESC` visually as a barh chart.

In [ ]:
### BEGIN SOLUTION

ax = ofns_ser.plot(kind = 'barh', color = 'blue')
ax.set_xlabel('Number of Calls')
ax.set_ylabel('Crime Category')
ax.set_title('Number of Calls By Crime Type')

### END SOLUTION

### 3.2 Getting a Better Plot
The plot above can be messy as it plots all offenses. Plot only the offenses that has more than 10000 calls

In [ ]:
### BEGIN SOLUTION
ofns_ser = result['OFNS_DESC'].value_counts().loc[lambda x: x > 10000]
ax = ofns_ser.plot(kind = 'barh', color = 'palevioletred')
ax.set_xlabel('Number of Calls')
ax.set_ylabel('Crime Category')
ax.set_title('Number of Calls By Crime Type')

### END SOLUTION

By contrast, the Seaborn library provides a specific function `countplot` built for plotting counts. It operates directly on the DataFrame itself i.e. there's no need to call `value_counts()` at all. This higher level approach makes it easier to work with. Use the y-label ("Crime Category"), x-label("Number of Calls") and title_of_plot("Number of Calls By Crime Type"

In [ ]:
### BEGIN SOLUTION

ax = sns.countplot(y = "OFNS_DESC", data = df)
ax.set_xlabel('Number of Calls')
ax.set_ylabel('Crime Category')
ax.set_title('Number of Calls By Crime Type')


### END SOLUTION

You may note that the ordering might be different for the seaborn plot (as compared to pandas plot). If we want the same ordering that we had in the pandas plot, we can use the order parameter of the `countplot` method. It takes a list of strings corresponding to the axis to be ordered. By passing the index of the `value_counts`, you can get the order you want.

In [ ]:
### BEGIN SOLUTION
ax = sns.countplot(y = 'OFNS_DESC',
              data = df,
              order = df['OFNS_DESC'].value_counts().sort_values().index)
ax.set_xlabel('Number of Calls')
ax.set_ylabel('Crime Category')
ax.set_title('Number of Calls By Crime Type')


### END SOLUTION

Now we have a pretty bar plot with the bars ordered by size. Though `seaborn` appears to provide a superior plot from a aesthetic point of view, the `pandas` plotting library is also good to understand. You'll get practice using both libraries in the following questions.

## An Additional Note on Plotting in Jupyter Notebooks

You may have noticed that many of our code cells involving plotting end with a semicolon (;). This prevents any extra output from the last line of the cell that we may not want to see. Try adding this to your own code in the following questions!

### 3.3 making more plots

Now it is your turn to make some more plots using `pandas` and `seaborn`. Let's start by looking at the distribution of calls over days of the week.

The CMPLNT_FR_DT field contains the date of the event. We would like to add a new column to the DataFrame that includes Day of the week (DAY_OF_WEEK) that indicates the day of the week. This can help us analyze the crimes on a specific day of the week. For example, we can answer questions such as "what day of the week that a LARSON is likely to happen in NYC?"


Add a new column `DAY_OF_WEEK` into the `calls` dataframe that has the day string (eg. 'Sunday') for the corresponding value in CMPLNT_FR_DT. For example, if the first 3 values of `CMPLNT_FR_DT` are `['01/27/2006, '01/28/2006, '01/29/2006]`, then the first 3 values of the `DAY_OF_WEEK` column should be `["Friday", "Saturday", "Sunday"]`.

**Hint:** *Try using the [Series.map](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.Series.map.html) function on `calls["OFNS_DESC"]`.  Can you assign this to the new column `calls["DAY_OF_WEEK"]`?*

In [ ]:
from datetime import datetime
import datetime
import calendar
days = {0:"Monday", 1:"Tuesday", 2:"Wednesday", 3:"Thursday", 4:"Friday", 5:"Saturday", 6:"Sunday"}
### BEGIN SOLUTION
from datetime import  date
df_1 = df
df_1['CMPLNT_FR_DT'] = pd.to_datetime(df_1['CMPLNT_FR_DT'], errors = 'coerce')
df_1['DAY_OF_WEEK'] = df_1['CMPLNT_FR_DT'].dt.weekday.map(days)

df_1
### END SOLUTION

### 3.4 Seaborn plots

Create a `seaborn` plot that shows the number of calls for each day of the week. You may want to use of the `rotation` argument in `ax.set_xticklabels`, which rotates the labels by 90 degrees.

In [ ]:
### BEGIN SOLUTION

order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
ax = sns.countplot(x = "DAY_OF_WEEK", data = df_1, order = order)
ax.set_xlabel('DAY_OF_WEEK')


ax.tick_params(axis='x', rotation=90)

ax.set_ylabel('Number of Calls')
ax.set_title('Number of Calls For Each Day of the Week')
### END SOLUTION

Now, let's make the same plot using `pandas`. Construct a vertical bar plot with the count of the number of calls (entries in the table) for each day of the week **ordered by the day of the week** (eg. `Sunday`, `Monday`, ...). Do not use `sns` for this plot. Be sure that your axes are labeled and that your plot is titled.

**Hint:** *Given a series `s`, and an array `coolIndex` that has the same entries as in `s.index`, `s[coolIndex]` will return a copy of the series in the same order as `coolIndex`.*

In [ ]:
### BEGIN SOLUTION


df_1['DAY_OF_WEEK'] = pd.Categorical(df_1['DAY_OF_WEEK'], categories=
    ['Sunday','Monday','Tuesday','Wednesday','Thursday','Friday', 'Saturday'],
    ordered=True)

calls_by_day = df_1['DAY_OF_WEEK'].value_counts()
calls_by_day = calls_by_day.sort_index()
ax = calls_by_day.plot(kind='bar')
ax.set_xlabel('DAY_OF_WEEK')
ax.set_ylabel('Number of Calls')
ax.set_title('Number of Calls For Each Day of the Week')


res = df_1['DAY_OF_WEEK'].value_counts()
res

### END SOLUTION

## 3.5 What Day of the Week is more calls?

Is it true that weekdays generally have slightly more calls than Saturday or Sunday? What can you say about the difference?

##### BEGIN SOLUTION

res = df_1['DAY_OF_WEEK'].value_counts()
res

Output:

Friday       112801
Wednesday    107742
Thursday     106424
Tuesday      106296
Saturday     106146
Monday       100440
Sunday        97713

Yes, weekdays generally have slightly more calls than Saturday or Sunday. 
On average, weekdays have ((100440 + 106296 + 107742 + 106424 + 112801)/5) = 106740.6 calls, which is more than the calls on Saturday or Sunday.


##### END SOLUTION

We can break down into some particular types of events to see their distribution. For example, let's make a bar plot for the OFNS_DESC "HARRASSMENT 2". Which day is the peak for "HARRASSMENT 2"?

This time, use `seaborn` to create a vertical bar plot of the number of total noise violations reported on each day of the week, again ordered by the days of the week starting with Sunday. Do not use `pandas` to plot.

**Hint:** *If you're stuck, use the code for the seaborn plot in above question as a starting point.*

In [ ]:
### BEGIN SOLUTION

har2_df = df_1[(df_1['OFNS_DESC'] == 'HARRASSMENT 2')]

order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
ax = sns.countplot(x = "DAY_OF_WEEK", data = har2_df, order = order)
ax.set_xlabel('DAY_OF_WEEK')
ax.tick_params(axis='x', rotation=90)
ax.set_ylabel('Number of Calls')
ax.set_title('Number of Harrassment 2 Calls For Each Day of the Week')

#For "HARRASSMENT 2", Friday is the peak.
### END SOLUTION

### 3.6 

Do you see anything interesting about the distribution of HARRASSMENT 2 calls over a week? Type a short answer below.
##### BEGIN SOLUTION

The distribution of HARRASSMENT 2 calls over a week is relatively stable, however, weekdays generally have slighly more calls than weekends. There is a drop from Friday to Saturday

##### END SOLUTION

### 3.7 More Plots

Let's look at a similar distribution but for a crime we have much more calls data about. In the cell below, create the same plot as you did in previous questions, but now looking at instances of the OFNS_DESC "BURGLARY" (instead of "HARRASSMENT 2"). Use either `pandas` or `seaborn` plotting as you desire.

In [ ]:
### BEGIN SOLUTION

burg_df = df_1[(df_1['OFNS_DESC'] == 'BURGLARY')]

order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
ax = sns.countplot(x = "DAY_OF_WEEK", data = burg_df, order = order)
ax.set_xlabel('DAY_OF_WEEK')
ax.tick_params(axis='x', rotation=90)
ax.set_ylabel('Number of Calls')
ax.set_title('Number of Burglary Calls For Each Day of the Week')

### END SOLUTION



### 3.8 time of events

Now let's look at the CMPLNT_TO_TM column which indicates the time for events. Since it contains hour and minute information, let's extract the hour info and create a new column named `Hour` in the `calls` dataframe. You should save the hour as an `int`. Then plot the frequency of each hour in the table (i.e., `value_counts()`) sorted by the hour of the day (i.e., `sort_index()`).

You will want to look into how to use:

* [Series.str.slice](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.Series.str.slice.html#pandas.Series.str.slice) to select the substring.
* [Series.astype](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.Series.astype.html) to change the type.

**Hint:** *The `str` helper member of a series can be used to grab substrings.  For example, `calls["CMPLNT_TO_TM"].str.slice(3,5)` returns the minute of each hour of the `CMPLNT_TO_TM`.*

In [ ]:
calls = df_1
calls["Hour"] = calls["CMPLNT_TO_TM"].str.slice(0,2).replace(np.NaN,0).astype('int')
calls

In [ ]:
### BEGIN SOLUTION


calls_by_hour = calls['Hour'].value_counts()
calls_by_hour = calls_by_hour.sort_index()
ax = calls_by_hour.plot(kind='bar')
ax.set_xlabel('Hour of the Day')
ax.set_ylabel('Number of Calls')
ax.set_title('Number of Calls Reporting Fraud For Each Day of the Week')


### END SOLUTION

Create a pandas bar plot showing the number of BURGLARY crimes committed at each hour of the day. Use the labels
* ax.set_xlabel("Hour of the Day")
* ax.set_ylabel("Number of Calls")
* ax.set_title("Number of Calls Reporting Fraud For Each Day of the Week");

In [ ]:
### BEGIN SOLUTION

burg_df = calls[(calls['OFNS_DESC'] == 'BURGLARY')]

calls_by_hour = burg_df['Hour'].value_counts()
calls_by_hour = calls_by_hour.sort_index()
ax = calls_by_hour.plot(kind='bar')
ax.set_xlabel('Hour of the Day')
ax.set_ylabel('Number of Calls')
ax.set_title('Number of Calls Reporting Fraud For Each Day of the Week')


### END SOLUTION

### 3.9 More plots

In the cell below, create a seaborn plot of the same data. Again, make sure you provide axes labels and a title for your plot.

In [ ]:
### BEGIN SOLUTION

burg_df = calls[(calls['OFNS_DESC'] == 'BURGLARY')]

ax = sns.countplot(x = "Hour", data = burg_df)
ax.tick_params(axis='x', rotation=90)
ax.set_xlabel('Hour of the Day')
ax.set_ylabel('Number of Calls')
ax.set_title('Number of Calls Reporting Fraud For Each Day of the Week')

### END SOLUTION

### 3.8 Spike in burglary?

According to your plots, is there a spike in calls reporting BURGLARY at any particular time? If so, Do you trust that this spike is legitimate, or could there be an issue with our data? Explain your reasoning in 1-2 sentences below.

#### BEGIN SOLUTION
The spike in burglary when directly looking at the plots seems to be at hour 0, aka 12 am. However, I don't trust that 
the spike is legitimate because our data has many NaN values which have been assigned hour 0. Therefore I think, the real spike occurs around hours 7 and 8.

#### END SOLUTION

In the cell below, we generate a boxplot which examines the hour of day of each crime broken down by the `OFNS_DESC` value.  To construct this plot we used the [DataFrame.boxplot](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.boxplot.html) documentation.

In [ ]:
calls.boxplot(column="Hour", by='OFNS_DESC', rot=90);

While the pandas boxplot is informative, we can use seaborn to create a more visually-appealing plot. Using seaborn, regenerate a better box plot. See either the textbook (https://www.textbook.ds100.org/ch/06/viz_quantitative.html) or the [seaborn boxplot documentation](https://seaborn.pydata.org/generated/seaborn.boxplot.html).

Looking at your plot, which crime type appears to have the largest interquartile range? Put your results into `answer` as a string.

In [ ]:
# Todo: Make a boxplot with seaborn
### BEGIN SOLUTION

ax = sns.boxplot(x="OFNS_DESC", y="Hour", data=calls)
ax.tick_params(axis='x', rotation=90)
ax.set_title('Distribution of Calls Over Hours in a Day, Grouped by Crime')

answer = "Prostitution & Related Offenses"

### END SOLUTION

### 3.9 - Visualization of crimes on a Map of NYC
finally we attempt to visualize the crimes committed in NYC on a Map. First we need to installing some mapping software. run the cell below to install folium package for mapping software.

In [ ]:
!pip install --upgrade folium

In [ ]:
buNYC = calls[calls["OFNS_DESC"] == "BURGLARY"][:20]
len(buNYC)

In [ ]:
### Plot the maps for BURGLARY in NYC
### if it takes too much time or map does not show up, try plotting a subset of the data set of for a specific crime
import folium
### BEGIN SOLUTION
#(40.835020524, -73.912060306)
map_burg = folium.Map(location = [40.835020524, -73.912060306], zoom_start = 4)

#plot locations
for (index,row) in buNYC.iterrows():
    folium.Marker(location = [row.loc['Latitude'], row.loc['Longitude']],
                 tooltip = 'click').add_to(map_burg)

    
#display map
map_burg
### END SOLUTION

In [ ]:
map1

 ## Congratulations !!!

<div class="alert alert-block alert-info">
<h2>Submission Instructions</h2> 
<b> Output:</b> Please **remove all output** from your notebook prior to submission<br>
<b> Data:</b> **DO NOT** submit any data files. You may lose points if you do so<br>
<b> File Name:</b> Please name the file as your_section_your_netID_Lab3.ipynb (eg. 01_adg133_Lab3.ipynb<br>
<b> Submit To: </b> Canvas &rarr; Assignments &rarr; Lab3 <br>
<b>Warning:</b> Failure to follow directions may result in loss of points.<br>
</div>

@2021 A.D. Gunawardena. Many people contributed to this lab including CS439 TA Liqin Long (currently @ Google). Much credit go to my friend and Princeton colleague Prof. Josh Hug (@ Berkeley), and Berkeley Data Science Group for their contributions to the original version. Please DO NOT share this lab and/or post them on public sites.